In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain.schema import Document

In [2]:
llm=ChatOllama(model="llama3.1")
embedding_model=OllamaEmbeddings(model="nomic-embed-text")

In [3]:
docs=PyPDFLoader("constitution_of_pakistan.pdf").load()

In [4]:
def clean_text(text):
    return " ".join(text.split())

In [5]:
from langchain.schema import Document
cleaned_docs = [Document(page_content=clean_text(doc.page_content)) for doc in docs]
for doc in cleaned_docs[:3]:
    print(f"{doc.page_content}\n")


TH E C O N STITU TIO N O F TH E ISL A M IC R EPU B L IC O F PA K ISTA N (A s am ended upto the Tw enty-sixth A m endm ent) G O V ER N M EN T O F PA K ISTA N M IN ISTR Y O F LAW A N D JU STIC E 2024

PREFACE This Thirteen Edition presents an updated text of the Constitution of the Islamic Republic of Pakistan, 1973 incorporating all the latest amendments. This latest version aims to serve as a valuable resource for the legal fraternity and the general public alike. The Law and Justice Division has also introduced a comprehensive database of the Federal Laws known as the “Pakistan code,” which provides easy access to the laws of Pakistan, both domestically and internationally. This database is accessible through a dedicated website and a user-friendly mobile applications, making it convenient for users to navigate and reference the laws of the country. I would like to express my appreciation for the officers of the Law and Justice Division who have contributed their efforts towards the p

In [6]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=2000,chunk_overlap=200)
documents=text_splitter.split_documents(cleaned_docs)

In [7]:
len(documents)

361

In [8]:
chroma_db=Chroma.from_documents(documents,embedding_model)

In [36]:
from langchain.prompts import ChatPromptTemplate

template = """
You are an expert legal assistant specializing in the Constitution of Pakistan (2024 Edition).

CONTEXT INFORMATION:
{context}

QUESTION: {question}

INSTRUCTIONS:
1. Parse the question to identify the precise constitutional provisions, principles, or mechanisms being queried.
2. Provide a structured, evidence-based response derived exclusively from the constitutional text provided in the context.
3. When citing specific provisions, use the standardized citation format: "Article X(Y)" for sections/clauses and "Part Z" for larger divisions.
4. For complex constitutional concepts, employ a hierarchical structure:
   - Primary heading: Constitutional principle/mechanism
   - Subheadings: Component elements
   - Bullet points: Specific provisions and their implications
5. If the question falls outside the scope of the provided constitutional text:
   - Clearly state the information gap
   - Identify the specific constitutional provisions that would be needed
   - Avoid speculative interpretation
6. Distinguish between:
   - Explicit constitutional text (direct quotations)
   - Constitutional mechanisms (procedural elements)
   - Constitutional principles (underlying concepts)

RESPONSE FORMAT:
CONSTITUTIONAL ANALYSIS: [Concise summary of the relevant constitutional framework]

DETAILED RESPONSE:
[Structured explanation with appropriate headings and citation-backed statements]

RELEVANT PROVISIONS: [Complete list of all constitutional articles, sections, and clauses referenced]

LIMITATIONS: [If applicable, note any constraints in addressing the question based on the provided context]
"""

prompt = ChatPromptTemplate.from_template(template)


In [37]:
retriever = chroma_db.as_retriever(
    search_type="mmr", 
    search_kwargs={
        "fetch_k": 15,  
        "k": 7,  
        "lambda_mult": 0.7,
    }
)


In [38]:
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
chain = ({"context": retriever, "question": RunnablePassthrough()}| prompt | llm| StrOutputParser())


In [41]:
def ask_question(question):
    response = chain.invoke(question)
    
    if not response.strip():
        return "The provided context does not contain the requested information."
    
    return response


question = "How does the Constitution prevent police brutality and unlawful arrests?"
answer = ask_question(question)
print(answer)


**CONSTITUTIONAL ANALYSIS:** Safeguards against Police Brutality and Unlawful Arrests

The Constitution provides several mechanisms to prevent police brutality and unlawful arrests. These safeguards can be categorized into explicit constitutional text, constitutional mechanisms, and underlying principles.

**I. Explicit Constitutional Text**

* Article 9: Protection against retrospective punishment
* Article 10: Protection against double punishment and self-incrimination
* Article 12: Inviolability of dignity of man, etc.
* Part 2: Principles of Policy (Article 29-40)

These articles explicitly address individual rights and protections against state actions.

**II. Constitutional Mechanisms**

* The Constitution establishes the Supreme Court as a guarantor of fundamental rights (Article 199).
* It provides for an independent judiciary to ensure that individual rights are protected (Article 197).

**III. Underlying Principles**

* Article 29: Principles of Policy - Promotion of social j